# 01 — Data acquisition

**Project:** *Selling Property Rental Ownership — A Hybrid Real Estate Model*  
**Author:** Dan Allouche · **Date:** May 2026

This notebook walks through every external series that feeds the calibration of the working paper:

1. **Notaires de France / INSEE** — quarterly Paris residential price index.
2. **INSEE IRL** — quarterly Indice de Référence des Loyers.
3. **FRED `IRLTLT01FRM156N`** — France 10-year sovereign yield (OAT).
4. **ECB SDW** — euro-area AAA 10Y spot rate (fallback / cross-check).
5. **FRED `CSUSHPISA`** — US Case-Shiller national index (out-of-sample sanity).
6. **Kenneth-French Library** — European 3-factor monthly returns.
7. **Visale / ANIL** — manual snapshot of yearly rental-arrears rates.

Every fetch reads the local snapshot under `data/raw/` by default. Pass `refresh=True` (or run `make data`) to hit the live upstream and rewrite the snapshot.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the in-repo package importable without an editable install.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from tranche_pricing.data import (
    SERIES,
    cache,
    ecb,
    fama_french,
    fred,
    insee_irl,
    notaires,
    oat,
    visale,
)
from tranche_pricing.viz.style import apply_style

apply_style()
pd.options.display.float_format = "{:,.3f}".format


## Load every series

Each call returns a tidy `DataFrame` with a `date` column and one or more series columns. We catch the `FileNotFoundError` that is raised when the local snapshot is missing so the notebook stays runnable even on a freshly cloned repo.

In [ ]:
loaded: dict[str, pd.DataFrame] = {}
missing: list[str] = []
for name, fetch_fn in SERIES.items():
    try:
        df = fetch_fn()
        loaded[name] = df
        head = df["date"].min().date() if "date" in df.columns and not df.empty else None
        tail = df["date"].max().date() if "date" in df.columns and not df.empty else None
        print(f"OK  {name:24s} rows={len(df):>5}  range={head} -> {tail}")
    except FileNotFoundError:
        missing.append(name)
        print(f"--  {name:24s} no snapshot yet (run `make data` to fetch).")

if missing:
    print("\nMissing series:", ", ".join(missing))
    print("Run `make data` (with internet access) to download them.")


## Provenance log

Every successful fetch records its upstream URL, row count and retrieval timestamp under `data/raw/_provenance.json`. This is what feeds the data appendix of the working paper.

In [ ]:
prov_path = cache.PROVENANCE_PATH
if prov_path.exists():
    import json
    with prov_path.open() as fh:
        prov = json.load(fh)
    prov_table = (
        pd.DataFrame(prov)
        .T.reset_index()
        .rename(columns={"index": "series"})
        .sort_values("series")
        .reset_index(drop=True)
    )
    display(prov_table)
else:
    print("No provenance file yet. Run a fetch with refresh=True to populate it.")


## Figure #1 — Paris residential price index

We plot the Notaires-INSEE quarterly index with French / euro-area recession bands overlaid. The inset shows the quarterly log-return series that drives the GBM and Merton calibrations for the GBM and Merton MLEs.

In [ ]:
from tranche_pricing.viz import figures

if "notaires_paris" in loaded:
    fig = figures.fig_paris_price_index(loaded["notaires_paris"])
    (ROOT / "artifacts/figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(ROOT / "artifacts/figures/fig_paris_price_index.pdf")
    fig.savefig(ROOT / "artifacts/figures/fig_paris_price_index.png", dpi=200)
    print("Saved fig_paris_price_index.{pdf,png} under artifacts/figures/.")
else:
    print("Notaires snapshot missing -- figure not produced.")
    print("Run `make data` to fetch the Notaires de France index.")


## Quick sanity tables

A compact summary of each series so the calibration step in `02_calibration.ipynb` starts from a known baseline.

In [ ]:
def _summary(name: str, df: pd.DataFrame, value_col: str) -> dict[str, object]:
    series = df[value_col].dropna()
    return {
        "series": name,
        "rows": int(len(series)),
        "min": float(series.min()) if not series.empty else float("nan"),
        "median": float(series.median()) if not series.empty else float("nan"),
        "max": float(series.max()) if not series.empty else float("nan"),
        "start": str(df["date"].min().date()) if not df.empty else "--",
        "end": str(df["date"].max().date()) if not df.empty else "--",
    }


summaries = []
mapping = {
    "notaires_paris": "price_index",
    "insee_irl": "irl",
    "oat_10y": "yield_pct",
    "ecb_yield_10y": "yield_pct",
    "case_shiller_us": "price_index",
    "visale": "arrears_rate",
}
for name, col in mapping.items():
    if name in loaded:
        summaries.append(_summary(name, loaded[name], col))

if summaries:
    display(pd.DataFrame(summaries).set_index("series"))


---
**Next step:** `02_calibration.ipynb` consumes the parquet under `data/processed/` to fit the GBM/Merton, Vasicek and copula parameters.